### Etapa 1: Leitura de arquivos

In [1]:
import pandas as pd

df_2021 = pd.read_csv("crime_2021.csv", sep=";", encoding="latin1", low_memory=False)
df_2022 = pd.read_csv("crime_2022.csv", sep=";", encoding="latin1", low_memory=False)
df_2023 = pd.read_csv("crime_2023.csv", sep=";", encoding="latin1", low_memory=False)
df_2024 = pd.read_csv("crime_2024.csv", sep=";", encoding="latin1", low_memory=False)
df_2025 = pd.read_csv("crime_2025.csv", sep=";", encoding="latin1", low_memory=False)
df_2026 = pd.read_csv("crime_2026.csv", sep=";", encoding="latin1", low_memory=False)

FileNotFoundError: [Errno 2] No such file or directory: 'crime_2021.csv'

In [ ]:
colunas_uteis = [
    "Sequência",
    "Data Fato",
    "Hora Fato",
    "Grupo Fato",
    "Tipo Enquadramento",
    "Tipo Fato",
    "Municipio Fato",
    "Local Fato",
    "Bairro",
    "Quantidade Vítimas",
    "Idade Vítima",
    "Sexo Vítima",
    "Cor Vítima",
]

dfs_por_ano = {
    2021: df_2021,
    2022: df_2022,
    2023: df_2023,
    2024: df_2024,
    2025: df_2025,
    2026: df_2026,
}

for ano, df in dfs_por_ano.items():
    df_filtrado = df[colunas_uteis]
    nome = f"df_{ano}_colunas_uteis.csv"
    df_filtrado.to_csv(nome, index=False, encoding="utf-8")
    print(f"Salvo: {nome} ({len(df_filtrado)} registros)")

Salvo: df_2021_colunas_uteis.csv (166150 registros)
Salvo: df_2022_colunas_uteis.csv (599648 registros)
Salvo: df_2023_colunas_uteis.csv (806945 registros)
Salvo: df_2024_colunas_uteis.csv (759848 registros)
Salvo: df_2025_colunas_uteis.csv (763795 registros)
Salvo: df_2026_colunas_uteis.csv (121882 registros)


### Etapa 2: Concatenando os dados

In [2]:
df_crimes = pd.concat([
    pd.read_csv(f"df_{ano}_colunas_uteis.csv", encoding="utf-8", low_memory=False)
    for ano in [2021, 2022, 2023, 2024, 2025, 2026]
], ignore_index=True)

print(f"Total de registros após concatenação: {len(df_crimes)}")

Total de registros após concatenação: 3218268


### Etapa 2.2: Padronização das colunas

In [3]:
df_crimes = df_crimes.rename(columns={
    "Sequência":          "sequencia",
    "Data Fato":          "data_fato",
    "Hora Fato":          "hora_fato",
    "Grupo Fato":         "grupo_fato",
    "Tipo Enquadramento": "tipo_enquadramento",
    "Tipo Fato":          "tipo_fato",
    "Municipio Fato":     "municipio_fato",
    "Local Fato":         "local_fato",
    "Bairro":             "bairro",
    "Quantidade Vítimas": "quantidade_vitimas",
    "Idade Vítima":       "idade_vitima",
    "Sexo Vítima":        "sexo_vitima",
    "Cor Vítima":         "cor_vitima",
})

### Etapa 2.3: Convertendo tipos dos dados

In [4]:
df_crimes["data_fato"] = pd.to_datetime(df_crimes["data_fato"], format="%d/%m/%Y", errors="coerce")

df_crimes["quantidade_vitimas"] = pd.to_numeric(df_crimes["quantidade_vitimas"], errors="coerce")
df_crimes["idade_vitima"]       = pd.to_numeric(df_crimes["idade_vitima"],       errors="coerce")
df_crimes["sequencia"]          = pd.to_numeric(df_crimes["sequencia"],          errors="coerce")

colunas_str = ["hora_fato", "grupo_fato", "tipo_enquadramento", "tipo_fato",
               "municipio_fato", "local_fato", "bairro", "sexo_vitima", "cor_vitima"]
for coluna in colunas_str:
    df_crimes[coluna] = df_crimes[coluna].astype("category")

print("Tipos de dados:")
print(df_crimes.dtypes)

print(f"\nTotal de registros: {len(df_crimes)}")
print(f"Período: {df_crimes['data_fato'].min()} até {df_crimes['data_fato'].max()}")
df_crimes.info()

Tipos de dados:
sequencia                    float64
data_fato             datetime64[ns]
hora_fato                   category
grupo_fato                  category
tipo_enquadramento          category
tipo_fato                   category
municipio_fato              category
local_fato                  category
bairro                      category
quantidade_vitimas           float64
idade_vitima                 float64
sexo_vitima                 category
cor_vitima                  category
dtype: object

Total de registros: 3218268
Período: 2021-10-01 00:00:00 até 2026-02-28 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3218268 entries, 0 to 3218267
Data columns (total 13 columns):
 #   Column              Dtype         
---  ------              -----         
 0   sequencia           float64       
 1   data_fato           datetime64[ns]
 2   hora_fato           category      
 3   grupo_fato          category      
 4   tipo_enquadramento  category      
 5   tipo_fato

## Etapa 3 – Limpeza e Padronização

### 3.1 – Verificando valores ausentes

In [5]:
print("Valores ausentes por coluna:")
print(df_crimes.isnull().sum())

Valores ausentes por coluna:
sequencia                  1
data_fato                  1
hora_fato                  0
grupo_fato                 0
tipo_enquadramento         0
tipo_fato                  1
municipio_fato             1
local_fato                 1
bairro                638855
quantidade_vitimas        17
idade_vitima          517817
sexo_vitima           519739
cor_vitima            517011
dtype: int64


In [6]:
# Colunas de texto: preenche com "Não informado"
for coluna in ["bairro", "hora_fato", "sexo_vitima", "cor_vitima"]:
    df_crimes[coluna] = (
        df_crimes[coluna]
        .astype(str)
        .replace("nan", "Não informado")
        .astype("category")
    )

# Colunas numéricas: mantém NaN (nem todo crime tem vítima identificada)

print("Valores ausentes após tratamento:")
print(df_crimes.isnull().sum())

Valores ausentes após tratamento:
sequencia                  1
data_fato                  1
hora_fato                  0
grupo_fato                 0
tipo_enquadramento         0
tipo_fato                  1
municipio_fato             1
local_fato                 1
bairro                     0
quantidade_vitimas        17
idade_vitima          517817
sexo_vitima                0
cor_vitima                 0
dtype: int64


### 3.2 – Removendo duplicatas

In [7]:
print(f"Registros antes de remover duplicatas: {len(df_crimes)}")

df_crimes = df_crimes.drop_duplicates()

print(f"Registros após remover duplicatas: {len(df_crimes)}")

Registros antes de remover duplicatas: 3218268
Registros após remover duplicatas: 3218268


### 3.3 – Formato de datas (YYYY-MM-DD)

In [8]:
df_crimes["data_fato"] = df_crimes["data_fato"].dt.strftime("%Y-%m-%d")

print("Exemplo de datas após padronização:")
print(df_crimes["data_fato"].head())

Exemplo de datas após padronização:
0    2021-10-01
1    2021-10-01
2    2021-10-01
3    2021-10-01
4    2021-10-01
Name: data_fato, dtype: object


### 3.4 – Inconsistências em colunas categóricas

In [9]:
colunas_categoricas = ["grupo_fato", "tipo_enquadramento", "tipo_fato",
                       "municipio_fato", "local_fato", "sexo_vitima", "cor_vitima", "bairro"]

for coluna in colunas_categoricas:
    df_crimes[coluna] = (
        df_crimes[coluna]
        .astype(str)
        .str.strip()
        .str.upper()
        .astype("category")
    )

print("Valores únicos em sexo_vitima:")
print(df_crimes["sexo_vitima"].unique())

print("\nValores únicos em cor_vitima:")
print(df_crimes["cor_vitima"].unique())

Valores únicos em sexo_vitima:
['MASCULINO', 'FEMININO', 'NÃO INFORMADO']
Categories (3, object): ['FEMININO', 'MASCULINO', 'NÃO INFORMADO']

Valores únicos em cor_vitima:
['BRANCA', 'PARDA', 'SEM INFORMAÇÃO', 'NÃO INFORMADO', 'PRETA', 'AMARELA', 'INDÍGENA']
Categories (7, object): ['AMARELA', 'BRANCA', 'INDÍGENA', 'NÃO INFORMADO', 'PARDA', 'PRETA', 'SEM INFORMAÇÃO']


### Checkpoint – Etapa 3

In [10]:
print(f"Total de registros: {len(df_crimes)}")
print(f"Total de colunas: {df_crimes.shape[1]}")
df_crimes.info()

df_crimes.to_csv("CrimesTratados.csv", index=False)
print("\nArquivo CrimesTratados.csv salvo com sucesso!")

Total de registros: 3218268
Total de colunas: 13
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3218268 entries, 0 to 3218267
Data columns (total 13 columns):
 #   Column              Dtype   
---  ------              -----   
 0   sequencia           float64 
 1   data_fato           object  
 2   hora_fato           category
 3   grupo_fato          category
 4   tipo_enquadramento  category
 5   tipo_fato           category
 6   municipio_fato      category
 7   local_fato          category
 8   bairro              category
 9   quantidade_vitimas  float64 
 10  idade_vitima        float64 
 11  sexo_vitima         category
 12  cor_vitima          category
dtypes: category(9), float64(3), object(1)
memory usage: 154.2+ MB

Arquivo CrimesTratados.csv salvo com sucesso!
